<a href="https://colab.research.google.com/github/FMZLAM2005/Lab4/blob/main/Untitled3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Task 1: Identify data quality issues

In [31]:
import pandas as pd
import numpy as np

df = pd.read_csv("student_performance.csv")
df.head()

,Gender,StudyHours,Attendance,PreviousGrade,FinalGrade
0,Female,5,90.0,85.0,88.0
1,Male,2,70.0,60.0,65.0
2,Female,8,95.0,92.0,94.0
3,Male,3,75.0,70.0,72.0
4,Female,6,85.0,80.0,83.0


In [25]:
print(df.dtypes)

Gender            object
StudyHours         int64
Attendance       float64
PreviousGrade    float64
FinalGrade       float64
dtype: object


In [26]:
print(df.isna().sum())

Gender           0
StudyHours       0
Attendance       1
PreviousGrade    1
FinalGrade       1
dtype: int64


In [27]:
print(df.describe())

       StudyHours  Attendance  PreviousGrade  FinalGrade
count   21.000000   20.000000      20.000000   20.000000
mean     5.476190   83.350000      79.650000   87.250000
std      3.124405   15.128485      16.010769   34.742474
min      0.000000   50.000000      40.000000   30.000000
25%      3.000000   71.500000      69.750000   70.250000
50%      6.000000   86.500000      83.000000   84.000000
75%      8.000000   95.000000      91.250000   91.500000
max     12.000000  110.000000     105.000000  200.000000


#Task 2: Apply one missing value strategy (Median)

In [28]:
df_task2 = df.copy()

df_task2['Attendance'] = df_task2['Attendance'].fillna(df_task2['Attendance'].median())
df_task2['PreviousGrade'] = df_task2['PreviousGrade'].fillna(df_task2['PreviousGrade'].median())
df_task2['FinalGrade'] = df_task2['FinalGrade'].fillna(df_task2['FinalGrade'].median())

print(df_task2.isna().sum())

Gender           0
StudyHours       0
Attendance       0
PreviousGrade    0
FinalGrade       0
dtype: int64


#Task 3: Detect and handle outliers using IQR

In [22]:
cols = ['StudyHours','Attendance','PreviousGrade','FinalGrade']

Q1 = df_task2[cols].quantile(0.25)
Q3 = df_task2[cols].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR

outliers = df_task2[(df_task2[cols] < lower) | (df_task2[cols] > upper)].any(axis=1)
print(df_task2[outliers])

df_task3 = df_task2[~outliers]

    Gender  StudyHours  Attendance  PreviousGrade  FinalGrade
15    Male          10       100.0           98.0       150.0
16  Female           0        50.0           40.0        30.0
17    Male          12       110.0          105.0       200.0


#Task 4: Normalize numerical features (Min-Max & Z-score)

In [23]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

num_cols = ['StudyHours','Attendance','PreviousGrade','FinalGrade']

scaler_minmax = MinMaxScaler()
df_minmax = df_task3.copy()
df_minmax[num_cols] = scaler_minmax.fit_transform(df_minmax[num_cols])
print("Min-Max normalized data:\n", df_minmax.head())

scaler_z = StandardScaler()
df_zscore = df_task3.copy()
df_zscore[num_cols] = scaler_z.fit_transform(df_zscore[num_cols])
print("Z-score standardized data:\n", df_zscore.head())

Min-Max normalized data:
    Gender  StudyHours  Attendance  PreviousGrade  FinalGrade
0  Female       0.500    0.789474          0.750    0.769231
1    Male       0.125    0.263158          0.125    0.179487
2  Female       0.875    0.921053          0.925    0.923077
3    Male       0.250    0.394737          0.375    0.358974
4  Female       0.625    0.657895          0.625    0.641026
Z-score standardized data:
    Gender  StudyHours  Attendance  PreviousGrade  FinalGrade
0  Female   -0.070535    0.627714       0.473622    0.678941
1    Male   -1.340157   -1.158666      -1.723593   -1.403144
2  Female    1.199088    1.074309       1.088842    1.222093
3    Male   -0.916949   -0.712071      -0.844707   -0.769466
4  Female    0.352673    0.181119       0.034179    0.226314


#Task 5: Apply PCA if numerical features show correlation

In [24]:
from sklearn.decomposition import PCA

corr = df_task3[num_cols].corr()
print(corr)

X = df_task3[num_cols]
pca = PCA(n_components=2)
principal_components = pca.fit_transform(X)

print("Explained Variance Ratio:", pca.explained_variance_ratio_)

               StudyHours  Attendance  PreviousGrade  FinalGrade
StudyHours       1.000000    0.975637       0.934354    0.954583
Attendance       0.975637    1.000000       0.928410    0.979149
PreviousGrade    0.934354    0.928410       1.000000    0.916504
FinalGrade       0.954583    0.979149       0.916504    1.000000
Explained Variance Ratio: [0.96048456 0.03224722]
